### 1. Identify the issues with the “Review” column in the UNITENReview.csv file

**Noise from URLs/HTML:** Text often contains web links or remnants of website tags.

**Irrelevant symbols:** Emojis, numbers, and punctuation marks that don't add semantic value.

**Inconsistency:** A mix of uppercase/lowercase and spelling mistakes.

**Chat Speak:** Heavy use of internet slang (lol, tbh) and contractions (don't, i'm).

**Redundancy:** Common "stopwords" (the, is, in) that bloat the dataset.

### 2. Perform the necessary text pre-processing steps based on the identified issues

In [1]:
import pandas as pd
import re
import emoji
import string
import nltk
from bs4 import BeautifulSoup
from autocorrect import Speller
from nltk.corpus import stopwords, wordnet
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
from nltk import pos_tag

# Setup
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger_eng')

spell = Speller(lang='en')
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

# Helper for POS tagging
def get_wordnet_pos(nltk_tag):
    if nltk_tag.startswith('J'): return wordnet.ADJ
    elif nltk_tag.startswith('V'): return wordnet.VERB
    elif nltk_tag.startswith('R'): return wordnet.ADV
    else: return wordnet.NOUN

# The Pre-processing Function
def preprocess_text(text):
    if not isinstance(text, str): return ""
    
    text = text.lower() # Step 1: Lowercase 
    text = re.sub(r'http\S+|www\S+', '', text) # Step 2: URLs 
    text = BeautifulSoup(text, "html.parser").get_text() # Step 3: HTML 
    text = emoji.replace_emoji(text, replace="") # Step 4: Emojis
    
    # Slang & Contractions (Assumes dictionaries defined previously) 
    # (In a real script, insert your slang_dict and contractions_dict here)
    
    text = text.translate(str.maketrans('', '', string.punctuation)) # Step 7 
    text = re.sub(r'\d+', '', text) # Step 8: Numbers 
    text = spell(text) # Step 9: Spelling 
    
    # Step 10: Stopwords
    text = " ".join([w for w in text.split() if w.lower() not in stop_words])
    
    # Step 11: Lemmatization
    tokens = word_tokenize(text)
    pos_tags = pos_tag(tokens)
    return " ".join([lemmatizer.lemmatize(w, get_wordnet_pos(t)) for w, t in pos_tags])

# Load and Apply 
df = pd.read_csv("UNITENReview.csv")
df["Processed_Review"] = df["Review"].apply(preprocess_text)

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\nabie\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\nabie\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\nabie\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\nabie\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\nabie\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
C:\Users\nabie\AppData\Local\Temp\ipykernel_15656\3374131057.py:37: MarkupResemblesLocatorWarning: The input looks more like a filename than markup

### 3. Save the result in a .csv file

In [2]:
# Saving the output [cite: 298]
df.to_csv("UNITEN_Processed_Results.csv", index=False)

# Display result [cite: 499]
print("File saved successfully!")
print(df[["Review", "Processed_Review"]].head())

File saved successfully!
                                              Review  \
0  Im happy with uniten actually, even the people...   
1  I’m having a pretty good time here, happy to m...   
2        a very neutral place in terms of everything   
3  I would say Uniten it's  a good university  bu...   
4   UNITEN is well-regarded, particularly for its...   

                                    Processed_Review  
0              im happy unite actually even people w  
1         i ’ m pretty good time happy meet w people  
2                      neutral place term everything  
3  would say united good university issue need im...  
4  united wellregarded particularly strong engine...  
